In [1]:
from os import getenv
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI(
    base_url="https://26120057-2478-resource.openai.azure.com/openai/v1",
    api_key=getenv("AZURE_API_KEY"),
)

MODEL = "gpt-5-nano"

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

Here’s how to run Ollama locally, per the course guidance:

1) Install Ollama
- macOS: download the .pkg and install it.
- Windows: download the .msi and install it.
- Linux: run in the terminal:
  curl -fsSL https://ollama.com/install.sh | sh

2) Run Ollama locally
- In a terminal, type:
  ollama run llama3
- This downloads the LLaMA 3 model (about 4 GB), starts the model locally, and opens a chat-like interface for you to ask questions.

3) Test the local Ollama server
- In a terminal, run:
  curl http://localhost:11434
- You should receive a JSON response similar to:
  {"models": [...]}

4) (Optional) Use the Python client
- Install the Python client:
  pip install ollama
- Minimal Python example:
  import ollama

  response = ollama.chat(
      model='llama3',
      messages=[{"role": "user", "content": your_prompt}]
  )

  print(response['message']['content'])


In [5]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model=MODEL,
    input=messages,
)

response.output_text

'Absolutely—usually you can join, but it depends on the course’s enrollment status, start dates, and any prerequisites.\n\nTo help you quickly, I need a bit more detail:\n- Which course and platform (university site, Coursera, Udemy, edX, etc.)?\n- Are there any prerequisites or a specific start date?\n- Is there a deadline or a waitlist?\n- Do you want credit or just access to materials?\n\nIf you can share the course name and platform, I can guide you step by step or even draft a quick message to the course team to ask about joining.\n\nIn the meantime, general steps to join:\n- Visit the course page and look for an “Enroll” or “Join” button.\n- Create an account or log in if you already have one.\n- Check prerequisites and start dates; note any deadlines.\n- Enroll and, if required, complete payment to access the course.\n- Confirm your enrollment and add the start date to your calendar.\n\nTell me the course details and I’ll tailor the guidance.'

In [6]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

In [9]:
function_calls = [item for item in response.output if item.type == 'function_call']
len(function_calls)

1

In [10]:
call = function_calls[0]

In [11]:
call

ResponseFunctionToolCall(arguments='{"query":"Can I join the course? how to enroll open enrollment late enrollment prerequisites"}', call_id='call_pqxB0YkqosnUn8CwbiIuvAIE', name='search', type='function_call', id='fc_00992607bc57c6c9006aef72f145fc81959f13f13eba7b678c', async_=None, caller=None, namespace=None, status='completed')

In [12]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course? how to enroll open enrollment late enrollment prerequisites'}

In [13]:
call.name

'search'

In [14]:
results = search(**args)

In [15]:
result_json = json.dumps(results, indent=2)

In [16]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [17]:
messages.extend(response.output)

In [18]:
messages.append(function_call_output)

In [19]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='rs_00992607bc57c6c9006aef72efbf3481958aaf1d36fe08e741', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq73Lx7OM_GdD3xtky6tP5RWfwNkBTwagO3QhZBTj0bW3OnLfVqQZ-TQPo2NwaunAwB6oPkLJlkyzQ_Flhb8L6EfppTSwpML5s_JKbxbK5-Thpz0kj4y5fpTASgpAjs2BvLs5cbt6W-gIXkBnqbkyelE2Nn4GgVsIH03iWM7SzbTr-qpWuQ7AoIESspkQtt5A2qtXikz07ftFV9XbFSVtSCHfCeh6Eo5pJNYNn1wxKpqU9MbdNE02MiCVVUuU6C_kF2S0V6q4nFREQDBNs6S_Zw27jrszGpuq2foERjVUir870wqhSuEQos8E_qeGQuEhzaCOklmTZQnJfD3TFswyRwKWKvEI02RdTP8TEepSYB5Cn6IaQ2Uu1-jIqazc74NSXVSWpvrBLGp4fPfg_RAwU-Ze2DS6FRxy-rjdpwCyyW8eAZSiDz8J_Jn0LfssRBmzlpY7idZaVX3BW9qqw7mlc5dwGFUyrWdnRSWlEqhFZgRwFbwDFwOIfzYFnvx1o4dpTpEnmILyQ0DZqCFAsCVm3gTZOKxMTpySh_xvCMiMl2IJ7jn8sLRpNOj52BeT5oGDSg0-OQ7zLZ5Lqhu7CrXsMDuE_STdjWsBW1NRx83aZA-1Re85SMgyfk3vI4wx-XyJRxxW0lZfKzly8ubykYQ891fV8y4Oi8SoqVZ8Oeazg6eHfiOQIWJkZn1vDINZes-CsjGoHgleB04U_PWzabtHlkHUH20BvNM4DxLrC-Sy7UWnHdvDbl5Pa4lg4Dyjckppb5i

In [20]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

In [21]:
print(response.output_text)

Yes—you can still join the llm-zoomcamp course.

Note: If you want to receive a course certificate, you’ll need to submit your project while submissions are still being accepted.

If you’d like, I can look up current enrollment deadlines or share the exact enrollment steps for this course. Tell me the course name you’re targeting (if it’s not llm-zoomcamp).


In [22]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(956, 609)

In [23]:
def calculate_gpt5nano_price(input_tokens, output_tokens):
    # Prices per 1M tokens (gpt-5-nano)
    INPUT_PRICE_PER_MILLION = 0.05   # $0.05 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.40  # $0.40 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt5nano_price(usage.input_tokens, usage.output_tokens)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0002914


In [24]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


In [25]:
response = openai_client.responses.create(
    model=MODEL,
    input=messages,
    tools=[search_tool]
)

response.output_text
print(response.output_text)

In [26]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [27]:
print([item for item in response.output if item.type == 'function_call'])

[ResponseFunctionToolCall(arguments='{"query":"course enrollment how to join FAQ"}', call_id='call_ZIa0NTHpRtCC3ONyabqUOMmZ', name='search', type='function_call', id='fc_0c641800ee4ebf14006aef72fad31881968de0214342cb8d10', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"Is enrollment open for this course FAQ"}', call_id='call_vQQMdL4WLw1caWGcZn63sD3k', name='search', type='function_call', id='fc_0c641800ee4ebf14006aef72fad3308196a3dd48ef3a8ca576', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"eligibility to enroll course FAQ"}', call_id='call_G7BXBaLS0991t6OXLiF6rPNu', name='search', type='function_call', id='fc_0c641800ee4ebf14006aef72fad33c8196b732491645b125ea', async_=None, caller=None, namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"auditing or auditing options for course FAQ"}', call_id='call_fCcdvZGqkGt5019iurQtYApX', name=

In [28]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"course enrollment how to join FAQ"}
function_call: search {"query":"Is enrollment open for this course FAQ"}
function_call: search {"query":"eligibility to enroll course FAQ"}
function_call: search {"query":"auditing or auditing options for course FAQ"}
function_call: search {"query":"start date deadlines enrollment course FAQ"}


In [29]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseReasoningItem(id='rs_0c641800ee4ebf14006aef72f7be3881969f098bdf60ce4b10', summary=[], type='reasoning', content=[], encrypted_content='gAAAAABq73L7SxB1G5qXUqwd_1NNB8lxFFVvgzHXaSwp-9H_vdpi4vTEeHpqbdcOnj4c3DFg_HWOUkVApRvPUCI--CK9aMrL5d7rfMK-XZSFWaj0kNDtUlI8zsNinKyUKSfOLk0_gb5WI8TKAw7-0aTihflbb9ZAGQVPQbCOrXantzJjrFuAxFkebUPDearfG7ZDfeVvxrYf0P3iHvL78vaqOi24t5N2xGIo08vbZOztKAT3_UZetrG_LdJPxLnE8tJn0zylL85aG

In [32]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it= 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model=MODEL,
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True
        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)

    it += 1

    if has_function_calls == False:
        break


iteration #1...
function_call: search {"query":"how to enroll in course open enrollment"}
function_call: search {"query":"late enrollment in course"}
function_call: search {"query":"is there a deadline to join course"}
function_call: search {"query":"course access for new students prerequisites"}
function_call: search {"query":"guest access to course materials"}
iteration #2...
ASSISTANT:
Yes. You can join the LLM Zoomcamp and start learning right away. Here’s how it works:

- You can start now: the videos, notebooks, and GitHub materials are available, and you can submit homework while the current live cohort is accepting submissions.
- Registration isn’t strictly required to begin—it's mainly for gauging interest. If you want a certificate, you’ll need to participate in a live cohort and complete the capstone project plus peer reviews.

What to do next:
- Check the current cohort and deadlines on the course management platform: https://courses.datatalks.club/llm-zoomcamp-2026/
- Star

In [33]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results
and then perform more searchers.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [35]:
def agent_loop(instructions, question, model="gpt-5-nano") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [36]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama local run"}
function_call: search {"query":"Olama install"}
function_call: search {"query":"Olama development environment"}
iteration #2...
ASSISTANT:
Sounds like you mean Ollama (the local LLM server). Here’s a quick guide to running Ollama locally.

What you’ll need
- A supported OS (macOS, Windows, or Linux)
- Sufficient disk space (models can be several GB)
- Internet access for the initial model download

Steps

1) Install Ollama
- macOS: download the .pkg from ollama.com/download and install.
- Windows: download the .msi and install.
- Linux: run the installer script
  curl -fsSL https://ollama.com/install.sh | sh

2) Run a local model
- Start a model (example with llama3):
  ollama run llama3
- This will download the LLaMA 3 model (~4 GB) and start the local server.

3) Verify the local server is running
- In a terminal, test the endpoint:
  curl http://localhost:11434
- You should get a JSON response listing the available mo

"Sounds like you mean Ollama (the local LLM server). Here’s a quick guide to running Ollama locally.\n\nWhat you’ll need\n- A supported OS (macOS, Windows, or Linux)\n- Sufficient disk space (models can be several GB)\n- Internet access for the initial model download\n\nSteps\n\n1) Install Ollama\n- macOS: download the .pkg from ollama.com/download and install.\n- Windows: download the .msi and install.\n- Linux: run the installer script\n  curl -fsSL https://ollama.com/install.sh | sh\n\n2) Run a local model\n- Start a model (example with llama3):\n  ollama run llama3\n- This will download the LLaMA 3 model (~4 GB) and start the local server.\n\n3) Verify the local server is running\n- In a terminal, test the endpoint:\n  curl http://localhost:11434\n- You should get a JSON response listing the available models.\n\n4) (Optional) Try using the Python client\n- Install the Python client:\n  pip install ollama\n- Example usage:\n  import ollama\n\n  response = ollama.chat(\n      model='

In [37]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening explanation"}
iteration #3...
ASSISTANT:
Do you mean The Queen's Gambit in chess, or the Queen's Gambit TV series on Netflix? Here are quick overviews of both:

Chess opening – Queen's Gambit
- What it is: A classic chess opening that starts with 1.d4 d5 2.c4. White offers the c-pawn to Black in hopes of gaining central control and better development.
- Key ideas:
  - Control the center (c4 aims to challenge Black’s d5 pawn).
  - Open lines for White’s pieces and establish a solid pawn structure.
  - If Black accepts the gambit (QGA), Black grabs the pawn but may have to defend it; if Black declines (QGD), Black supports the center and players continue normal development.
- Main branches:
  - Queen's Gambit Declined (QGD): 1.d4 d5 2.c4 e6 — very solid, traditional setup.
  - Queen's Gambit Accepted (QGA): 1.d4 d5 2.c4 dxc4 — Black tak

"Do you mean The Queen's Gambit in chess, or the Queen's Gambit TV series on Netflix? Here are quick overviews of both:\n\nChess opening – Queen's Gambit\n- What it is: A classic chess opening that starts with 1.d4 d5 2.c4. White offers the c-pawn to Black in hopes of gaining central control and better development.\n- Key ideas:\n  - Control the center (c4 aims to challenge Black’s d5 pawn).\n  - Open lines for White’s pieces and establish a solid pawn structure.\n  - If Black accepts the gambit (QGA), Black grabs the pawn but may have to defend it; if Black declines (QGD), Black supports the center and players continue normal development.\n- Main branches:\n  - Queen's Gambit Declined (QGD): 1.d4 d5 2.c4 e6 — very solid, traditional setup.\n  - Queen's Gambit Accepted (QGA): 1.d4 d5 2.c4 dxc4 — Black takes the pawn, White often aims to regain it with strong piece activity.\n  - Slav Defense: 1.d4 d5 2.c4 c6 — a very solid, popular alternative that keeps a strong center.\n- Typical fol

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I couldn’t find any FAQ entry in our course materials about the Queen’s Gambit. It looks like this question isn’t covered by the course FAQ and may be off-topic for this course.

If you’d like, I can help with course-related topics (deadlines, assignment details, where to find resources, how to submit work, etc.). Or tell me what specific aspect you want to explore, and I’ll help within the scope of the course.

Are there other areas you’d like to explore?


'I couldn’t find any FAQ entry in our course materials about the Queen’s Gambit. It looks like this question isn’t covered by the course FAQ and may be off-topic for this course.\n\nIf you’d like, I can help with course-related topics (deadlines, assignment details, where to find resources, how to submit work, etc.). Or tell me what specific aspect you want to explore, and I’ll help within the scope of the course.\n\nAre there other areas you’d like to explore?'